# 02 - Plate Recognition Pipeline

Plate detection (`best.pt`) -> `PlateRecognizer` (char.pt raw + LPSR-enhanced + TrOCR) -> grammar/state validation -> temporal consensus across frames. Reads plates from the first demo video found, or from any JPEG in `/content`.


## Setup


In [ ]:
import os, sys, pathlib

# --- point these at the real locations on Colab -----------------------------
AI_DIR = os.environ.get("TRAFFIQ_AI_DIR", "/content/traffIQ/ai")
WEIGHTS_DIR = os.environ.get("TRAFFIQ_WEIGHTS_DIR", "")

# /content/implementation is where notebook 00 wrote the modules (or your
# cloned/Mounted repo if you prefer to import from there instead).
sys.path.insert(0, "/content/implementation")
os.environ["TRAFFIQ_AI_DIR"] = str(AI_DIR)

from pipeline.config import Config, colab_config

if WEIGHTS_DIR:
    cfg = colab_config(weights_dir=WEIGHTS_DIR, ai_dir=AI_DIR)
else:
    cfg = Config(ai_dir=AI_DIR)
cfg.validate(require_all=False)
print("ai_dir      :", cfg.ai_dir)
print("ref_repo    :", cfg.ref_repo_dir)
print("plate_weights:", cfg.plate_weights)
print("vehicle_weights:", cfg.vehicle_weights)


In [ ]:
import cv2, glob, pathlib
from adapters.plate_detector import PlateDetector
from adapters.plate_recognition import PlateRecognizerAdapter
from adapters.temporal_filter import TemporalPlateFilter

plate_det = PlateDetector(cfg)
rec = PlateRecognizerAdapter(cfg)
temporal = TemporalPlateFilter(cfg)


## 1. Collect plates from a short burst of frames


In [ ]:
VIDEO = glob.glob("/content/[Hh]ighway*.mp4") or glob.glob("/content/**/[Hh]ighway*.mp4", recursive=True)
IMAGES = glob.glob("/content/**/*.jpg", recursive=True) + glob.glob("/content/**/*.png", recursive=True)
frames = []
if IMAGES:
    frames = [(i, cv2.imread(IMAGES[0])) for i in [0]]
elif VIDEO:
    cap = cv2.VideoCapture(VIDEO[0])
    want = 15
    n = 0
    while n < want:
        ok, f = cap.read()
        if not ok:
            break
        if n % 1 == 0 or True:
            if n % (want // 3 or 1) == 0:
                frames.append((cap.get(cv2.CAP_PROP_POS_FRAMES), f))
        n += 1
    cap.release()
print("using", len(frames), "frames from", (VIDEO if VIDEO else IMAGES)[:1])


## 2. Detect + recognize + vote


In [ ]:
import pandas as pd
rows, burstdets = [], []
for frame_no, frame in frames:
    dets = []
    for p in plate_det.detect(frame):
        crop = plate_det.crop(frame, p["bbox"])
        if crop.size == 0:
            continue
        r = rec.recognize(crop)
        dets.append({"bbox": p["bbox"], "text": r["text"], "score": r["score"],
                     "source": r["source"]})
        rows.append({"frame": int(frame_no), "bbox": p["bbox"], "text": r["text"],
                     "confidence": r["confidence"], "format_valid": r["format_valid"],
                     "source": r["source"], "raw": r["raw_char_text"],
                     "sr": r["sr_char_text"], "trocr": r["trocr_text"]})
    burstdets.extend(temporal.update(dets))
df = pd.DataFrame(rows)
print(f"{len(df)} plate detections")
df.head(10)


## 3. Stats: read rate / grammar validity


In [ ]:
if not df.empty:
    read = df[df.text != ""]
    print("detections            :", len(df))
    print("non-empty reads       :", len(read))
    print("grammar-valid reads   :", int(read.format_valid.sum()))
    print("unique texts          :", read.text.nunique())
    print(read.source.value_counts().to_dict())
    read[["text", "confidence", "format_valid", "source"]].head(15)


## 4. Confidence histogram


In [ ]:
import matplotlib.pyplot as plt
if not df.empty:
    read = df[df.text != ""]
    ax = read.confidence.plot.hist(bins=20, title="Plate confidence (heuristic, clamped 0-1)")
    ax.set_xlabel("confidence"); ax.set_ylabel("count")
    plt.show()
else:
    print("no plate detections to plot")


## 5. Temporal consensus on a per-plate burst


In [ ]:
from collections import Counter
texts = [d.get("consensus_text") or "" for d in burstdets if d.get("consensus_text")]
if texts:
    print("consensus texts per plate-box track:", Counter(texts))
else:
    print("no consensus texts on this burst (no repeated tracked plate)")
